<a href="https://colab.research.google.com/github/Hrishik1033/FedAVG-Implementation/blob/main/FedAVG(Non_IID_Partitioning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train.shape

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


(60000, 28, 28)

# Load the datasets

In [2]:
def load_mnist():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0
    x_train = x_train.reshape(-1, 28, 28, 1)
    x_test = x_test.reshape(-1, 28, 28, 1)
    return (x_train, y_train.astype("int64")), (x_test, y_test.astype("int64"))

# Implementing Dirichlet Distribution

In [3]:
import numpy as np
def partition_non_iid_unequal(y, num_clients, alpha=0.5, min_size=10):
    """Using Dirichlet Function for Non IID Partitioning"""
    n = len(y)
    idx = np.random.permutation(n)

    while True:
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))# Makes a np array of size num_clients, with different proportions whose sum is 1
        sizes = (proportions * n).astype(int)# Converting the proportions to the data sizes

        sizes[-1] = n - sizes[:-1].sum()# Making sure that the sum is exactly the size of the dataset
        if sizes.min() >= min_size:
            break

    client_indices = []
    start = 0
    for size in sizes:
        client_indices.append(idx[start : start+size])
        start+=size

    print("Non-IID (unequal quantity) split sizes per client:",
          [len(c) for c in client_indices])
    return client_indices

# Building the model and compiling it

In [5]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Input
from keras.models import Model

def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, 5, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(64, 5, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])

    input = Input(shape=(28, 28, 1))
    x = Conv2D(32, 5, padding="same", activation="relu")(input)
    x = MaxPooling2D(2)(x)
    x = Conv2D(64, 5, padding="same", activation="relu")(x)
    x = MaxPooling2D(2)(x)
    x = Flatten()(x)
    x = Dense(512, activation="relu")(x)
    output = Dense(10, activation="softmax")(x)
    model = Model(inputs=input, outputs=output)
    return model

In [6]:
def compile_model(model,lr):
  model.compile(
      loss="sparse_categorical_crossentropy",
      optimizer=tf.keras.optimizers.SGD(learning_rate=lr),
      metrics=["accuracy"],
  )
  return model

# Gettin the weights and setting weights

In [7]:
def get_weights(model):
  return [w for w in model.get_weights()]

In [8]:
def set_weights(model,weights):
  model.set_weights(weights)

# Update the clients after weights are averaged

In [9]:
def client_update(global_weights, x_k, y_k, E, B, lr, debug=False):
    local_model = compile_model(build_model(), lr)
    set_weights(local_model, global_weights)
    batch_size = len(x_k) if B is None else B   # B=None means "full batch", i.e. B=infinity in the paper
    hist = local_model.fit(x_k, y_k, epochs=E, batch_size=batch_size, verbose=1)
    if debug:
        final_loss = hist.history["loss"][-1]
        final_acc = hist.history["accuracy"][-1]
        print(f"      client n={len(x_k):4d} | local final loss {final_loss:.3f} "
              f"| local final acc {final_acc:.3f}")
    return get_weights(local_model), len(x_k) # Getting the updated weights from the client and also the lenght of the client

# Doing the federated averaging

In [10]:
def federated_average(client_weights_list, client_sizes):
    total = sum(client_sizes)
    avg = [np.zeros_like(w) for w in client_weights_list[0]]
    for weights, n_k in zip(client_weights_list, client_sizes):
        for i, w in enumerate(weights):
            avg[i] += (n_k / total) * w
    return avg

In [11]:
def run_fedavg(x_train, y_train, x_test, y_test, client_indices,
               num_rounds=60, C=0.6, E=1, B=32, lr=0.01, log_every=1,
               debug_clients=False):
    K = len(client_indices)# Total clients
    m = max(int(C * K), 1)   # number of clients sampled each round

    global_model = compile_model(build_model(), lr)
    global_weights = get_weights(global_model)

    history = []
    for t in range(1, num_rounds + 1):
        #server selects a random fraction C of clients and the same client is not repeated
        selected = np.random.choice(K, m, replace=False)

        if debug_clients:
            print(f"  Round {t}: sampling {m} clients -> {selected.tolist()}")

        client_weights_list, client_sizes = [], []
        for k in selected:
            idxs = client_indices[k] # The client_indices store the index range for each and every client
            x_k, y_k = x_train[idxs], y_train[idxs]

            # each selected client trains locally and reports back
            w_k, n_k = client_update(global_weights, x_k, y_k, E, B, lr,
                                      debug=debug_clients)
            client_weights_list.append(w_k)
            client_sizes.append(n_k)

        #server aggregates: weighted average of client models
        global_weights = federated_average(client_weights_list, client_sizes)
        set_weights(global_model, global_weights)

        if t % log_every == 0 or t == num_rounds:
            loss, acc = global_model.evaluate(x_test, y_test, verbose=0)
            print(f"Round {t:3d}/{num_rounds} | clients used: {m:3d} "
                  f"| test loss: {loss:.4f} | test acc: {acc:.4f}")
            history.append((t, loss, acc))

    return global_model, history


In [12]:
if __name__ == "__main__":
    print("Loading MNIST...")
    (x_train, y_train), (x_test, y_test) = load_mnist()

    NUM_CLIENTS = 5

    print("\n--- FedAvg on non-IID (unequal quantity) data, 5 clients ---")
    non_iid_clients = partition_non_iid_unequal(y_train, NUM_CLIENTS, alpha=0.5)
    # debug_clients=True for the first few rounds is the fastest way to
    # confirm clients are individually learning even while the aggregate
    # accuracy is still low early on.
    run_fedavg(x_train, y_train, x_test, y_test, non_iid_clients,
               num_rounds=60, C=0.6, E=1, B=32, lr=0.01,
               debug_clients=True)


Loading MNIST...

--- FedAvg on non-IID (unequal quantity) data, 5 clients ---
Non-IID (unequal quantity) split sizes per client: [15330, 5421, 31544, 3666, 4039]
  Round 1: sampling 3 clients -> [3, 4, 2]
115/115 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.3044 - loss: 2.1970
      client n=3666 | local final loss 2.197 | local final acc 0.304
127/127 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.3283 - loss: 2.1709
      client n=4039 | local final loss 2.171 | local final acc 0.328
986/986 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.8165 - loss: 0.6519
      client n=31544 | local final loss 0.652 | local final acc 0.817
Round   1/60 | clients used:   3 | test loss: 0.2644 | test acc: 0.9403
  Round 2: sampling 3 clients -> [2, 4, 3]
986/986 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9440 - loss: 0.1906
      client n=31544 | local final loss 0.191 | local final acc 0.944
127/127 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.9364 - loss: 0.2375
      client n=4039 | l